# 09 · Maintenance and capstone project

**Time:** ~40 min · **You will learn:** keep tables healthy (compaction, snapshot expiry) and combine everything in a small pipeline.

## Part A: Table maintenance
Frequent small writes create many tiny files and many snapshots. Both slow reads and cost storage, so tables need regular care.

In [ ]:
from datetime import datetime
from helpers import get_spark, full_name

spark = get_spark("09-maintenance")
t = full_name("clicks")
spark.sql(f"DROP TABLE IF EXISTS {t} PURGE")
spark.sql(f"CREATE TABLE {t} (id BIGINT, page STRING, ts TIMESTAMP) USING iceberg")

# 12 tiny appends, like a streaming job committing every few seconds
for i in range(12):
    spark.createDataFrame([(i, f"/page/{i % 3}", datetime(2024, 4, 1, i))], "id long, page string, ts timestamp") \
         .writeTo(t).append()

def stats():
    files = spark.sql(f"SELECT COUNT(*) FROM {t}.files").collect()[0][0]
    snaps = spark.sql(f"SELECT COUNT(*) FROM {t}.snapshots").collect()[0][0]
    print(f"files={files} snapshots={snaps} rows={spark.table(t).count()}")
    return files, snaps

files_before, snaps_before = stats()

### Compaction: `rewrite_data_files`
Merges many small files into fewer, larger ones (a new snapshot; data is unchanged).

In [ ]:
spark.sql(f"CALL demo.system.rewrite_data_files(table => 'course.clicks')").show(truncate=False)
files_after, _ = stats()
assert files_after < files_before

### Expire snapshots
Old snapshots keep old files alive. Expiring them frees storage but ends time travel to those versions.

In [ ]:
cutoff = spark.sql("SELECT current_timestamp() + INTERVAL 1 MINUTE").collect()[0][0].strftime("%Y-%m-%d %H:%M:%S")
spark.sql(f"CALL demo.system.expire_snapshots(table => 'course.clicks', older_than => TIMESTAMP '{cutoff}', retain_last => 1)").show(truncate=False)
_, snaps_after = stats()
assert snaps_after == 1

### Orphan files
Files left behind by failed jobs (written but never committed) can be found and deleted with `remove_orphan_files`:
```sql
CALL demo.system.remove_orphan_files(table => 'course.clicks', dry_run => true)
```
`dry_run => true` only lists what would be deleted. We do not run it here because this image has no Hadoop `s3://`
file system (the procedure lists the bucket through Hadoop, unlike the other procedures). It works as-is on
setups with `s3a` configured.

In production run these on a schedule (for example nightly): compact, expire snapshots older than your retention window
(7 days is a common default), then remove orphan files.

---
## Part B: Capstone, a mini clickstream pipeline
**Goal:** ingest events with PySpark, analyse them with PyIceberg + DuckDB, fix a data problem with time travel.

In [ ]:
import random
from datetime import timedelta
from pyspark.sql import functions as F

ev = full_name("web_events")
spark.sql(f"DROP TABLE IF EXISTS {ev} PURGE")
spark.sql(f"""
    CREATE TABLE {ev} (event_id BIGINT, user_id BIGINT, page STRING, duration_ms INT, event_ts TIMESTAMP)
    USING iceberg PARTITIONED BY (days(event_ts))
""")

rng = random.Random(42)
start = datetime(2024, 5, 1)
rows = [(i, rng.randint(1, 100), rng.choice(["/home", "/pricing", "/docs", "/blog"]),
         rng.randint(50, 5000), start + timedelta(minutes=17 * i)) for i in range(2000)]
spark.createDataFrame(rows, "event_id long, user_id long, page string, duration_ms int, event_ts timestamp") \
     .writeTo(ev).append()
print("ingested", spark.table(ev).count(), "events")

**Step 1.** Aggregate with PySpark.

In [ ]:
(spark.table(ev).groupBy(F.to_date("event_ts").alias("day"))
     .agg(F.count("*").alias("events"), F.avg("duration_ms").alias("avg_ms"))
     .orderBy("day").show(5))

**Step 2.** A bad batch arrives (negative durations). Load it, then notice.

In [ ]:
bad = [(10_000 + i, 1, "/home", -1, datetime(2024, 5, 20, 12, i)) for i in range(5)]
spark.createDataFrame(bad, "event_id long, user_id long, page string, duration_ms int, event_ts timestamp").writeTo(ev).append()
print("negative durations:", spark.table(ev).filter("duration_ms < 0").count())

**Step 3.** Fix it with time travel: roll back to the snapshot before the bad batch.

In [ ]:
hist = spark.sql(f"SELECT snapshot_id FROM {ev}.history ORDER BY made_current_at").collect()
spark.sql(f"CALL demo.system.rollback_to_snapshot('course.web_events', {hist[-2].snapshot_id})")
print("negative durations after rollback:", spark.table(ev).filter("duration_ms < 0").count())
assert spark.table(ev).filter("duration_ms < 0").count() == 0

**Step 4.** Hand over to the analysts: query the same table from PyIceberg + DuckDB, no Spark needed.

In [ ]:
from pyiceberg.catalog import load_catalog

events = load_catalog("default").load_table("course.web_events")
con = events.scan(selected_fields=("page", "duration_ms")).to_duckdb(table_name="web_events")
con.execute("""
    SELECT page, COUNT(*) AS views, ROUND(AVG(duration_ms)) AS avg_ms
    FROM web_events GROUP BY page ORDER BY views DESC
""").fetchdf()

**Step 5.** Evolve the schema as requirements change, then clean up.

In [ ]:
spark.sql(f"ALTER TABLE {ev} ADD COLUMN device STRING")
spark.sql(f"UPDATE {ev} SET device = 'mobile' WHERE user_id % 2 = 0")
spark.sql(f"SELECT device, COUNT(*) AS events FROM {ev} GROUP BY device ORDER BY device").show()

### Your turn

Clean up: drop every table created in this course (`orders`, `events`, `clicks`, `web_events`, `customers`, `products`)
with `DROP TABLE ... PURGE`, and verify with PyIceberg that `course.list_tables` is empty.

In [ ]:
# Write your solution here


<details>
<summary>Show solution</summary>

```python
for name in ["orders", "events", "clicks", "web_events", "customers", "products"]:
    spark.sql(f"DROP TABLE IF EXISTS demo.course.{name} PURGE")
assert load_catalog("default").list_tables("course") == []
```

</details>

## 🎓 You made it
You can now create and evolve Iceberg tables with PySpark, inspect their internals, time travel, merge changes,
and read and write them from plain Python with PyIceberg.

**Where to go next**
* The bundled notebooks in Jupyter: *Iceberg - Table Maintenance Spark Procedures*, *Write-Audit-Publish (WAP) with Branches*, *PyIceberg - Write support*.
* Official docs: https://iceberg.apache.org/docs/latest/ and https://py.iceberg.apache.org/
* Try the Flink example in `flink-example/` to stream data into the same kind of table.